# Matter by ecosystem: who supports what, and how far the test evidence reaches

**Claim cluster:** `C007`, `C008`, `C009`, `C010`, `C011`, `C012`, `C013`, `C014`, `C015`, `C016`, `C017`, `C018`, `C097`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ('C007', 'C008', 'C009', 'C010', 'C011', 'C012', 'C013', 'C014', 'C015', 'C016', 'C017', 'C018', 'C097')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

The claims below are unevenly evidenced on purpose: `C008` and `C011` are documented facts about specification content, while `C016`, `C017` and `C097` are reported signals from third-party device testing. The panels show which is which before any cell of the matrix is coloured, because a capability map built from reported signal is not the same object as one built from conformance tests.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### The corpus has no support matrix, so this one is built from claims

There is no per-ecosystem capability table in the database. What exists instead is a set of claims, each naming an ecosystem and a capability in prose. The figure therefore encodes *claims*, not support: every coloured cell carries the id of the claim it comes from, and a cell no claim covers is left blank and marked `no claim in this cluster records this cell`. Blank is not the same as absent, and the figure never implies it is.

In [ ]:
# The cell readings are an explicit, three-point reading of the claim text:
#   1   the claim states the capability is present
#   0.5 the claim states it with a recorded qualification
#   0   the claim states it is absent, failing, or slower
# A cell with no entry stays unrecorded. The mapping is written out in full so
# that each reading can be checked against the claim text shown on hover.
ECOSYSTEMS = ["Apple", "Google", "Amazon", "Samsung SmartThings"]
CAPABILITIES = ["Matter cameras", "In-platform automation rules", "Multi-admin"]

CELLS = {
    ("Matter cameras", "Samsung SmartThings"): ("C008", 1),
    ("Matter cameras", "Amazon"): ("C009", 1),
    ("Matter cameras", "Apple"): ("C010", 0),
    ("Matter cameras", "Google"): ("C010", 0),
    ("In-platform automation rules", "Samsung SmartThings"): ("C014", 1),
    ("In-platform automation rules", "Amazon"): ("C014", 0),
    ("In-platform automation rules", "Apple"): ("C014", 0),
    ("In-platform automation rules", "Google"): ("C014", 0),
    ("Multi-admin", "Apple"): ("C017", 1),
    ("Multi-admin", "Amazon"): ("C016", 0.5),
    ("Multi-admin", "Samsung SmartThings"): ("C097", 0.5),
}

_claims_by_id = claims.set_index("local_id")
_rows, _texts, _hovers = [], [], []
for _capability in CAPABILITIES:
    _row, _row_text, _row_hover = [], [], []
    for _ecosystem in ECOSYSTEMS:
        _cid, _status = CELLS.get((_capability, _ecosystem), (None, None))
        _row.append(_status)
        _row_text.append(_cid or "")
        if _cid is None:
            _row_hover.append(f"{_ecosystem}<br>{_capability}"
                              "<br>no claim in this cluster records this cell")
            continue
        _claim = _claims_by_id.loc[_cid]
        _row_hover.append(
            f"{_ecosystem}<br>{_capability}<br><b>{_cid}</b>"
            f" [{_claim['claim_type']}, {_claim['confidence']} confidence]"
            f"<br>{_claim['claim_text']}"
        )
    _rows.append(_row)
    _texts.append(_row_text)
    _hovers.append(_row_hover)

fig = go.Figure(go.Heatmap(
    z=_rows,
    x=ECOSYSTEMS,
    y=CAPABILITIES,
    zmin=0,
    zmax=1,
    colorscale=[[0.0, "#e8b6bb"], [0.5, "#f3e2b8"], [1.0, "#bcd9c4"]],
    xgap=3,
    ygap=3,
    text=_texts,
    texttemplate="%{text}",
    textfont=dict(size=12),
    customdata=_hovers,
    hovertemplate="%{customdata}<extra></extra>",
    colorbar=dict(
        title="claim reading",
        tickmode="array",
        tickvals=[0, 0.5, 1],
        ticktext=["absent / slower", "qualified", "present"],
    ),
))
fig.update_layout(
    title=("Ecosystem capability map, one claim per cell "
           "(blank = no claim in this cluster)"),
    height=380,
    xaxis=dict(side="top"),
    yaxis=dict(autorange="reversed"),
)
fig.show()

## Supporting context

### How far does the evidence behind the map actually reach?

Two claims bound the test evidence: `C013` states that Samsung SmartThings supports 58 Matter device types while other platforms 'may support fewer', and `C097` reports an Allion Labs functional test of multi-admin. The metrics below record the scale of that test inventory (`M086`) and the minimum device inventory the corpus's own interoperability protocol calls for (`M087`). `M004` is the only per-ecosystem device-type count anywhere in the corpus, so the breadth chart has exactly one counted bar and three unrecorded ones.

In [ ]:
display(HTML(
    "<h4>The test inventory behind the interoperability claims</h4>"
    "<p><code>M086</code> is the scale of the Allion Labs functional test; "
    "<code>M087</code> is the minimum device inventory the corpus's own "
    "interoperability protocol proposes. Both are read from the database.</p>"
))
_inventory = viz.metrics_by_id(["M004", "M005", "M086", "M087"])
show(_inventory[["local_id", "metric_name", "value", "unit", "confidence",
                 "source_ref", "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))

# M004 is the only per-ecosystem device-type count recorded anywhere in this
# corpus. C013 states that other platforms "may support fewer", which is a
# statement about the absence of a count, not a count of zero, so the other
# three bars are labelled unrecorded and drawn at zero height.
_types = viz.metrics_by_id(["M004"])
_samsung = viz.numeric_value(_types["value"].iloc[0]) if not _types.empty else None
_breadth = pd.DataFrame([
    {"ecosystem": "Samsung SmartThings", "device_types": _samsung,
     "reading": "counted in M004"},
    {"ecosystem": "Amazon", "device_types": None,
     "reading": "no count recorded; C013 says 'may support fewer'"},
    {"ecosystem": "Google", "device_types": None,
     "reading": "no count recorded; C013 says 'may support fewer'"},
    {"ecosystem": "Apple", "device_types": None,
     "reading": "no count recorded; C013 says 'may support fewer'"},
])
show(_breadth)
fig = go.Figure(go.Bar(
    x=[value if value is not None else 0 for value in _breadth["device_types"]],
    y=_breadth["ecosystem"],
    orientation="h",
    marker_color=["#1f77b4" if value is not None else "#d9d9d9"
                  for value in _breadth["device_types"]],
    text=["%g" % value if value is not None else "not recorded"
          for value in _breadth["device_types"]],
    textposition="outside",
    customdata=list(zip(_breadth["reading"], _breadth["device_types"])),
    hovertemplate="%{y}<br>%{customdata[0]}<extra></extra>",
))
fig.update_layout(
    title=("Matter device types supported, as recorded: one counted value, "
           "three ecosystems unrecorded (M004, C013)"),
    xaxis_title="device types",
    height=300,
    yaxis=dict(autorange="reversed"),
)
fig.show()
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### What the matrix is not

- **It is a claim map, not a test report.** Ten of the thirteen claims in this cluster are `reported signal` from third-party testing, one is an `inference` (`C015`: certification confirms conformance but does not guarantee ecosystem compatibility), and only two (`C008`, `C011`) are documented facts about specification content rather than about shipping behaviour.
- **Two claims cover several ecosystems each.** `C010` ('Apple and Google have been slower to implement camera support than Samsung and Amazon') and `C014` ('Amazon, Apple, and Google lack native support for creating automation rules ... while Samsung supports it') are single sentences mapped to several cells. The same claim id appears in each of those cells so the shallowness of the attribution stays visible on screen.
- **Only one ecosystem has a device-type count.** `M004` counts Samsung SmartThings; the corpus records no equivalent for Apple, Google or Amazon, so a four-row parity matrix would be invented. That is why wave 1 deferred this notebook and why this one runs on claims instead.
- **An unresolved conformance question sits inside the cluster.** `C015` says certification does not guarantee compatibility and `C097` records first-pairing success followed by a failure to operate the paired device. Both are kept; neither is averaged away.
- **`C017` carries no falsifier** (`falsifier_stated = false`), so the claim that Apple's HomePod Mini 'exhibited robust multi-admin support' has no recorded observation that would disconfirm it.

In [ ]:
# Claims that cannot be placed in a matrix cell without inventing an
# attribution: they are ecosystem-agnostic, or describe a failure mode rather
# than a capability. Listing them is the honest alternative to colouring a cell
# the corpus never filled.
_agnostic = ["C007", "C011", "C012", "C015", "C018"]
display(HTML(
    "<h4>Cluster claims deliberately left out of the matrix</h4>"
    "<p>Each claim below applies to several ecosystems, or describes "
    "behaviour rather than a capability.</p>"
))
show(claims.loc[claims["local_id"].isin(_agnostic),
                ["local_id", "claim_type", "confidence", "falsifier_missing",
                 "claim_text"]].rename(
    columns={"local_id": "claim", "claim_type": "type", "confidence": "conf",
             "falsifier_missing": "no_falsifier"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Claim text, evidence class, confidence and falsifier status come from `viz.fetch_claims`; the test-inventory numbers from `viz.metrics_by_id` by explicit local id. The cell readings are notebook-level data, printed as a table above and cross-checked claim by claim in the provenance panels, so the mapping is inspectable even though it is authored here rather than stored in the corpus.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()